<a href="https://colab.research.google.com/github/RGarancs/applied-ai-academy-labs/blob/main/dundermifflinfull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Dunder Mifflin — the whole order book, two countries

**Applied AI Academy — problem set with worked answers**

**Used in:** Lesson 8 — AI runs on data plumbing, so readiness matters more than model choice  
**Rows:** 18,393 order lines + 800 return rows  ·  **Files:** `dunder_mifflin_us_orders.csv`, `dunder_mifflin_ca_orders.csv`, `dunder_mifflin_returns.csv`

---
### What this lab is
The case (AIA-C61) is about an analyst who produced a fast, confident, arithmetically
flawless group profit number that was wrong in six separate ways. Not one of the
errors was a calculation error. Every one was a correct calculation over a column
whose meaning nobody had established.

This notebook is the same job, done properly. Run it in order. Each problem is
stated first, then solved, then the answer is spelled out.

> Read `dunder_mifflin.GUIDE.md` before you start, and then check whether the data
> agrees with it. In several places it does not.


## Setup


In [ ]:
%matplotlib inline
import pandas as pd, numpy as np, os, urllib.request
import matplotlib, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Applied AI Academy chart style — calm, legible when projected
INK, ACCENT, CORE, BUILDER, DANGER = "#26214a", "#544d94", "#5b8a6e", "#a8845c", "#b05a5a"
matplotlib.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9c4d8", "axes.labelcolor": INK, "axes.titlesize": 13,
    "axes.titleweight": "600", "axes.titlecolor": INK, "axes.titlepad": 14,
    "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "font.size": 10, "grid.color": "#e6e2ef", "axes.grid": True,
    "axes.axisbelow": True, "grid.linewidth": .8, "figure.facecolor": "white",
})
BASE = "https://appliedai.center/assets/datasets/"
FILES = {
    "us":      "dunder_mifflin_us_orders.csv",
    "ca":      "dunder_mifflin_ca_orders.csv",
    "returns": "dunder_mifflin_returns.csv",
}
for name in FILES.values():
    if not os.path.exists(name):
        print("Downloading", name)
        urllib.request.urlretrieve(BASE + name, name)

us  = pd.read_csv(FILES["us"], low_memory=False)
ca  = pd.read_csv(FILES["ca"], low_memory=False)
ret = pd.read_csv(FILES["returns"], low_memory=False)
print("US      ", us.shape)
print("Canada  ", ca.shape)
print("Returns ", ret.shape)


---
## Problem 1 · Easy — read the two column lists before you add anything

The two order books were built by different teams at different times. Before you
compute a single group number, find out where they disagree.


In [ ]:
a, b = set(us.columns), set(ca.columns)
print("shared      :", sorted(a & b))
print()
print("US only     :", sorted(a - b))
print()
print("Canada only :", sorted(b - a))


**Answer.** Three things matter here and all three change the group number.

1. Quantity is called `Quantity` in the US and `Order Quantity` in Canada. Stack the
   two frames and you get two half-empty columns rather than one full one.
2. Canada has no `Country` column at all. After a concatenation there is nothing in
   the row that says which country it came from unless you put it there.
3. Canada carries `Shipping Cost` and `Investment and Admin costs per unit`, which the
   US book does not. That is the warning that `Profit` may not mean the same thing on
   both sides. Problem 4 tests it.


---
## Problem 2 · Easy — the number the analyst reported

Add the two `Sales` columns together and the two `Profit` columns together, the way
the case says it was done. Write the number down. You are going to take it apart.


In [ ]:
naive_sales  = us["Sales"].sum()  + ca["Sales"].sum()
naive_profit = us["Profit"].sum() + ca["Profit"].sum()
print(f"group sales  {naive_sales:>14,.0f}")
print(f"group profit {naive_profit:>14,.0f}")
print()
print(f"US     sales {us['Sales'].sum():>14,.0f}   profit {us['Profit'].sum():>12,.0f}")
print(f"Canada sales {ca['Sales'].sum():>14,.0f}   profit {ca['Profit'].sum():>12,.0f}")


**Answer.** Both totals are wrong, and neither is wrong because of the arithmetic.
Canada is denominated in Canadian dollars. Nothing in the file says so: there is no
currency column, and the header just says `Sales`. Adding it to a US dollar column is
adding two different units, which a spreadsheet will do without complaint.

The habit to build: **ask what a column is denominated in before you add it to another
one.** It is the same question in every currency, every unit and every rate.


---
## Problem 3 · Medium — convert, and say what you assumed

Restate the group number in one currency. There is no rate in the files, so you have
to supply one, and the important part is that you write down which one and why.


In [ ]:
# The period is 2015-2018. A flat 0.77 USD per CAD is the average over it.
# It is an assumption, it is stated, and it can be argued with. That is the point.
CAD_TO_USD = 0.77

us_sales, ca_sales = us["Sales"].sum(), ca["Sales"].sum() * CAD_TO_USD
us_prof,  ca_prof  = us["Profit"].sum(), ca["Profit"].sum() * CAD_TO_USD
print(f"US      sales {us_sales:>13,.0f}  profit {us_prof:>12,.0f}")
print(f"Canada  sales {ca_sales:>13,.0f}  profit {ca_prof:>12,.0f}   (converted at {CAD_TO_USD})")
print(f"GROUP   sales {us_sales + ca_sales:>13,.0f}  profit {us_prof + ca_prof:>12,.0f}")
print()
print(f"Canada was overstated by {ca["Sales"].sum() - ca_sales:,.0f} USD in the naive total.")


**Answer.** The correction is a fifth off every Canadian figure. Note what it does to
the *ranking* as well as the totals: any recommendation that rested on Canada looking
larger than it is now has to be re-argued.

A number you cannot state the assumptions behind is not a finding, it is a guess with
formatting. Write the rate and the reason next to the number.


---
## Problem 4 · Medium — do the two books mean the same thing by "profit"?

Canada carries two cost columns the US does not. Test whether Canadian `Profit` has
already had them taken off, or not.


In [ ]:
cols = ["Sales", "Profit", "Shipping Cost", "Investment and Admin costs per unit", "Order Quantity"]
sample = ca[cols].head(8).copy()
sample["implied_costs"] = sample["Shipping Cost"] + sample["Investment and Admin costs per unit"] * sample["Order Quantity"]
sample["sales_minus_profit"] = sample["Sales"] - sample["Profit"]
print(sample.to_string(index=False))
print()
print("If the two right-hand columns tracked each other, Canadian profit would already")
print("be net of those costs. Compare them.")


**Answer.** They do not track. Canadian `Profit` is computed on a different cost base
from US `Profit`, so the two columns are not the same measure and adding them produces
a number that measures nothing.

This is the error that is hardest to catch and easiest to explain: it needed no
arithmetic at all, only reading the two column lists side by side and asking why one
country carries costs the other does not.


---
## Problem 5 · Medium — the returns file will multiply your revenue

Join returns to orders and check the row count before and after.


In [ ]:
print("returns rows          :", len(ret))
print("distinct order ids    :", ret["Order ID"].nunique())

naive = us.merge(ret, on="Order ID", how="left")
print("\nUS rows before join   :", len(us))
print("US rows after naive   :", len(naive), " <-- revenue now counted more than once")

clean = us.merge(ret.drop_duplicates("Order ID"), on="Order ID", how="left")
clean["Returned"] = clean["Returned"].fillna("No")
print("US rows after dedupe  :", len(clean))

returned = clean[clean["Returned"] == "Yes"]
print(f"\nreturned lines {len(returned):,} · sales on them {returned['Sales'].sum():,.0f}"
      f" · profit on them {returned['Profit'].sum():,.0f}")


**Answer.** The returns file has 800 rows and 296 distinct order numbers, so a plain
join duplicates every order that appears more than once, and every duplicated row
brings its sales and its profit with it.

The check is one line and it is the habit worth keeping: **count the rows before and
after a join.** If the number went up, the join is wrong.


---
## Problem 6 · Advanced — test an explanation instead of accepting one

The case has the analyst explain a pattern in Canadian order quantities by saying the
Canadian business sells in case packs of twelve. It sounds right. Test it.


In [ ]:
q = ca["Order Quantity"].dropna()
share12 = (q % 12 == 0).mean()
print(f"share of Canadian quantities divisible by 12: {share12:.1%}")
print(f"expected if quantities were spread evenly 1-50: {1/12:.1%}")
print("\nmost common quantities:")
print(q.value_counts().head(8).to_string())


**Answer.** 8.3 per cent, against the 8.3 per cent you would expect from quantities
spread evenly with no case packs at all. The two numbers are the same. There is no
case-pack effect in this data, and the explanation is dead.

This one matters more than it looks. A fabricated citation has no referent and dies
the moment somebody checks it, so the failure is loud and cheap to find. A plausible
explanation for a **real** pattern has a referent, satisfies the reader's need for a
reason, and survives the meeting. Test explanations, not just facts.


---
## Problem 7 · Advanced — the group number, defensibly

Now produce the number the board asked for, with every decision written down beside it.


In [ ]:
u = us.rename(columns={"Quantity": "units"}).copy()
c = ca.rename(columns={"Order Quantity": "units"}).copy()
u["country"], c["country"] = "United States", "Canada"
for col in ("Sales", "Profit"):
    c[col] = c[col] * CAD_TO_USD          # decision 1: one currency, stated rate

keep = ["country", "Order ID", "Order Date", "Segment", "Category", "Sub-Category",
        "Sales", "Profit", "Discount", "units"]
grp = pd.concat([u[keep], c[keep]], ignore_index=True)

r = ret.drop_duplicates("Order ID").assign(returned=True)[["Order ID", "returned"]]
grp = grp.merge(r, on="Order ID", how="left")     # decision 2: dedupe before joining
grp["returned"] = grp["returned"].fillna(False)

out = grp.groupby("country").agg(sales=("Sales", "sum"), profit=("Profit", "sum"),
                                 lines=("Sales", "size"))
out["margin"] = out["profit"] / out["sales"]
print(out.to_string(formatters={"sales": "{:,.0f}".format, "profit": "{:,.0f}".format,
                                "margin": "{:.1%}".format}))
print(f"\nGROUP sales {grp['Sales'].sum():,.0f} · profit {grp['Profit'].sum():,.0f}"
      f" · returned lines {int(grp['returned'].sum()):,}")


**Answer, and the thing to hand in.** The number matters less than the list beside it:

1. Canadian figures converted at 0.77 USD per CAD, a period average. Stated, arguable.
2. Returns deduplicated on order number before joining, because the file is per item.
3. Quantity columns reconciled by renaming, not by stacking two half-empty ones.
4. **Canadian and US profit still sit on different cost bases.** This is not fixed
   here, because it cannot be fixed from these files. It has to be asked of the
   Canadian finance team, and until it is answered the group margin is indicative.

Point 4 is the honest one, and it is what separates a defensible number from a
confident one. Say what you could not establish.


---
## What to take away

Every error in this lab was a correct calculation over a misunderstood column. That
failure mode does not get better as models get better, because the model is not doing
the arithmetic wrong. It is doing exactly what it was asked, to data whose meaning
nobody established.

Four habits, and they cost about a minute each:

- Read the column lists of two files side by side before you combine them.
- Ask what every measure is denominated in.
- Count the rows before and after a join.
- Test the explanation, not just the number.


---
## The shipping promise

Every shipping mode is a promise. Same Day means today. First Class means one
to two days. Second Class means three to four, and Standard means up to a week.
The order date and the ship date are both in the file, so the promise can be
checked against what actually happened.

Two questions, and the second one is a trap that catches most people:

1. **Do we ship when we say we will?**
2. **Do late orders get returned more often?** Compute it at line level first,
   then at order level, and explain what you see before reading on.

In [ ]:
# 1 · promise against delivery
us["Order Date"] = pd.to_datetime(us["Order Date"])
us["Ship Date"]  = pd.to_datetime(us["Ship Date"])
us["days"] = (us["Ship Date"] - us["Order Date"]).dt.days

PROMISE = {"Same Day": 0, "First Class": 2, "Second Class": 4, "Standard Class": 7}
us["late"] = us["days"] > us["Ship Mode"].map(PROMISE)

o = us.groupby("Order ID").agg(mode=("Ship Mode", "first"), late=("late", "first"))
late_pct = (100 * o.groupby("mode")["late"].mean()).reindex(
    ["First Class", "Second Class", "Same Day", "Standard Class"])

fig, ax = plt.subplots(figsize=(8, 4))
cols = [DANGER if v >= 20 else BUILDER if v >= 4 else CORE for v in late_pct]
ax.bar(late_pct.index, late_pct.values, color=cols, width=.6)
for i, v in enumerate(late_pct.values):
    ax.text(i, v + .8, f"{v:.0f}%", ha="center", fontweight="bold")
ax.set_ylabel("orders shipped later than promised, %")
ax.set_title("The premium promise is the broken one", loc="left")
plt.tight_layout(); plt.show()

print("Promised: Same Day = today, First Class = 1-2 days,")
print("          Second Class = 3-4 days, Standard = up to 7 days.")
print("Standard is never late because promising a week is barely promising anything.")

In [ ]:
# 2 · the correlation trap: late orders and returns
returned = set(ret["Order ID"])
us["returned"] = us["Order ID"].isin(returned)

line_rates = us.groupby("late")["returned"].mean() * 100

ord_lvl = us.groupby("Order ID").agg(late=("late", "first"), mode=("mode", "first")
                                     if "mode" in us else ("Ship Mode", "first"),
                                     returned=("returned", "first"))
order_rates = ord_lvl.groupby("late")["returned"].mean() * 100
fc = ord_lvl[ord_lvl["mode"] == "First Class"]
fc_rates = fc.groupby("late")["returned"].mean() * 100

import numpy as np
labels = ["by LINE\n(wrong unit)", "by ORDER\n(right unit)", "First Class\norders only"]
on_time = [line_rates[False], order_rates[False], fc_rates[False]]
late    = [line_rates[True],  order_rates[True],  fc_rates[True]]
x = np.arange(3); w = .35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, on_time, w, color=CORE, label="on time")
ax.bar(x + w/2, late,   w, color=DANGER, label="late")
for i in range(3):
    ax.text(i - w/2, on_time[i] + .2, f"{on_time[i]:.1f}", ha="center")
    ax.text(i + w/2, late[i] + .2,   f"{late[i]:.1f}",   ha="center")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("share returned, %")
ax.legend(frameon=False)
ax.set_title("The gap shrinks as the unit of analysis gets more honest", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# 3 · the same check on the Canadian book, which turns out to be impossible
ca["Order Date"] = pd.to_datetime(ca["Order Date"])
ca["Ship Date"]  = pd.to_datetime(ca["Ship Date"])
ca["days"] = (ca["Ship Date"] - ca["Order Date"]).dt.days

print("Canadian ship modes :", sorted(ca["Ship Mode"].unique()))
print("US ship modes       :", sorted(us["Ship Mode"].unique()))
print()
print("Different words entirely, and no Canadian mode states a promise,")
print("so there is nothing to be late against. Instead, Canada has a")
print("priority field. Check whether it does anything:")
print()
pri = ca.groupby("Order Priority").agg(
    days=("days", "mean"),
    express_share=("Ship Mode", lambda s: (s == "Express Air").mean()),
    freight=("Shipping Cost", "mean"))
print(pri.round(2))

In [ ]:
# 4 · priority and Express, drawn
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
order = ["Critical", "High", "Medium", "Not Specified", "Low"]
means = ca.groupby("Order Priority")["days"].mean().reindex(order)
cols = [DANGER, ACCENT, ACCENT, ACCENT, BUILDER]
a1.bar(order, means.values, color=cols, width=.6)
for i, v in enumerate(means.values):
    a1.text(i, v + .06, f"{v:.1f}", ha="center", fontweight="bold")
a1.set_ylabel("days from order to ship")
a1.set_title("Critical and Not Specified arrive on the same day", loc="left")
a1.tick_params(axis="x", rotation=15)

m = ca.groupby("Ship Mode").agg(days=("days", "mean"), freight=("Shipping Cost", "mean"))
x = np.arange(2); w = .35
a2.bar(x - w/2, [m.loc["Express Air", "days"], m.loc["Express Air", "freight"]], w,
       color=BUILDER, label="Express Air")
a2.bar(x + w/2, [m.loc["Regular Air", "days"], m.loc["Regular Air", "freight"]], w,
       color=ACCENT, label="Regular Air")
a2.set_xticks(x); a2.set_xticklabels(["days to ship", "freight $/unit"])
a2.legend(frameon=False)
a2.set_title("Express buys nothing", loc="left")
plt.tight_layout(); plt.show()

**Answer.** The first question has a clean finding: First Class breaks its one-to-two-day
promise on **four orders in ten**, Second Class on one in five, and Standard is never late,
because a one-week promise is barely a promise. The company charges for speed and does not
deliver it, and no margin dashboard will ever show that, because lateness does not touch profit.

The second question is the lesson. At line level, late orders look **45 per cent** more likely
to be returned: 11.1 against 7.6. Recut at order level it is 7.6 against 5.7, inside the noise
for a sample this size, and inside First Class alone it is 7.3 against 6.6, which is nothing.
Returns happen to **orders**, and a big order counts many times in a line-level cut, so if big
orders are slower to pick and also more likely to have one item returned, the correlation
appears without lateness causing anything.

Same lesson as the returns sheet: **the unit of analysis decides the finding.** Pick it before
you compute, or the data will pick a story for you.

And the Canadian book cannot even be asked the first question. Its modes are different words,
none carries a promise, and its priority field is a wish: Critical, High, Medium and Not
Specified all ship in about a day and a half, only Low differs, and Express Air matches
Regular Air on speed and freight to within three cents. One order in eight pays for a label.